# V12 Phase 1V — entry episode and journey-count accounting

## tl;dr

At one funded FAST-run episode per observation, V10 has 197 positive and 482 negative episodes. Excluding 57 tail episodes, the ordinary path totals -423.25R and the +1/-1 count curve has a -342 balance. Applying multi-speed ownership only at the first actual entry saves 65 first-Child stops and 16 repeated first stops, but loses 70 positive episodes and 16 tail episodes. The ordinary count slope remains almost unchanged (-0.550 to -0.546).

## Context & Methods

The decision unit is one funded FAST-run entry episode at its first actual-entry Child. This is a proxy for an economically distinct journey start, not proof of the complete larger market Journey. Every episode contributes +1 for positive total R, -1 for negative total R, or 0, regardless of Child count, weight, or R magnitude. Later Children remain inside admitted episodes but are reported separately.

### Key assumptions

- Phase-1I run IDs are used only as the closest existing entry-episode boundary.
- A tail episode is counted once when existing >=5R tail units are present.
- The policy uses only the first Child's causal Phase-1U ownership state.
- All evidence is consumed development history and has no action authority.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

repo = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'output').is_dir() and (path / 'research' / 'v12').is_dir())
pack = repo / 'output' / 'v12_phase1v_entry_episode_accounting_20260926_a'
ledger = pd.read_csv(pack / 'V12_PHASE1V_POLICY_LEDGER.csv', parse_dates=['run_start'])
cards = pd.read_csv(pack / 'V12_PHASE1V_POLICY_SCORECARDS.csv').set_index('policy')
states = pd.read_csv(pack / 'V12_PHASE1V_STATE_SCORECARDS.csv')
roles = pd.read_csv(pack / 'V12_PHASE1V_OUTCOME_ROLE_SCORECARDS.csv')
blocks = pd.read_csv(pack / 'V12_PHASE1V_INTER_TAIL_BLOCKS.csv', parse_dates=['start', 'end'])
quality = json.loads((pack / 'V12_PHASE1V_DATA_QUALITY.json').read_text(encoding='utf-8'))
assert quality['common_episode_rows'] == 679
assert quality['missing_ownership_state'] == 0
assert quality['episode_decomposition_max_R_error'] < 1e-10
cards[['episodes', 'first_child_hard_sl_episodes', 'repeat_first_child_stop_episodes', 'positive_episodes', 'negative_episodes', 'tail_episodes', 'non_tail_count_slope', 'non_tail_net_R', 'net_R_units']]

## Data

The Phase-1I run ledger and Phase-1K Child decomposition reconcile to numeric precision. There are 679 first Children and 730 later Children, with no duplicate run IDs, missing ownership states, signal mismatches, decision-time mismatches, or direction mismatches.

In [ ]:
control = ledger[ledger.policy == 'V10_ENTRY_EPISODE_CONTROL'].sort_values('run_start')
primary = ledger[ledger.policy == 'ENTRY_OWNERSHIP_AUTHORIZED'].sort_values('run_start')
control_blocks = blocks[blocks.policy == 'V10_ENTRY_EPISODE_CONTROL'].sort_values('block_id')

plt.rcParams.update({'font.size': 10, 'axes.titlesize': 12, 'axes.labelsize': 10})
fig, axes = plt.subplots(3, 1, figsize=(12, 10), constrained_layout=True)

axes[0].plot(control.run_start, control.cumulative_net_R, color='#24577a', linewidth=1.8, label='Actual cumulative R')
tail = control[control.tail_episode == 1]
axes[0].scatter(tail.run_start, tail.cumulative_net_R, color='#d6a53a', s=24, zorder=3, label='Tail episode')
axes[0].axhline(0, color='#555555', linewidth=0.8)
axes[0].set_title('Actual V10 episode equity is lifted by sparse tail episodes')
axes[0].set_ylabel('Cumulative R')
axes[0].legend(loc='upper left', frameon=False)
axes[0].grid(axis='y', color='#dddddd', linewidth=0.6)

axes[1].plot(control.run_start, control.cumulative_episode_count_score, color='#b45f36', linewidth=1.6, label='V10 all episodes')
axes[1].plot(control.run_start, control.cumulative_non_tail_count_score, color='#777777', linewidth=1.4, linestyle='--', label='V10 non-tail episodes')
axes[1].plot(primary.run_start, primary.cumulative_non_tail_count_score, color='#24577a', linewidth=1.5, label='Entry policy non-tail episodes')
axes[1].axhline(0, color='#555555', linewidth=0.8)
axes[1].set_title('One-vote-per-episode curves remain persistently downward')
axes[1].set_ylabel('Cumulative +1 / -1 score')
axes[1].legend(loc='lower left', frameon=False, ncol=3)
axes[1].grid(axis='y', color='#dddddd', linewidth=0.6)

bar_colors = ['#24577a' if value > 0 else '#b8b8b8' for value in control_blocks.net_R]
axes[2].bar(control_blocks.block_id, control_blocks.net_R, color=bar_colors, width=0.8)
axes[2].axhline(0, color='#555555', linewidth=0.8)
axes[2].set_title('Net R between tail episodes: 7 positive blocks and 46 negative blocks')
axes[2].set_xlabel('Chronological inter-tail block')
axes[2].set_ylabel('Block net R')
axes[2].grid(axis='y', color='#dddddd', linewidth=0.6)

chart_path = pack / 'V12_PHASE1V_CUMULATIVE_CURVES.png'
fig.savefig(chart_path, dpi=160, facecolor='white')
plt.show()
chart_path

## Results

The actual-R curve and count curve answer different questions. Actual R is positive because 57 tail episodes contribute enough magnitude to offset a -423.25R non-tail path. When every episode receives one equal vote, the balance is 197 positive versus 482 negative. The ownership policy reduces first-entry stops but does not change the ordinary negative-episode process.

In [ ]:
states[['ownership_state', 'episodes', 'first_child_hard_sl_rate', 'positive_episodes', 'negative_episodes', 'tail_episodes', 'non_tail_count_slope', 'non_tail_net_R_per_episode']].sort_values('first_child_hard_sl_rate')

In [ ]:
roles[['grouping', 'role', 'episodes', 'first_child_hard_sl_episodes', 'positive_episodes', 'negative_episodes', 'tail_episodes', 'first_child_R', 'later_child_R', 'net_R_units']]

## Takeaways

1. The user's accounting correction is valid: one large journey must count once when judging entry quality.
2. First-Child Hard SL is a clean failure label: 144 of 145 such episodes finish negative; only one later recovers positive.
3. It is not the whole ordinary decline: 338 negative episodes occur without a first-Child Hard SL, and 48 begin with a positive first Child before later participation turns the episode negative.
4. Every multi-speed ownership state still has a negative non-tail count slope. Current FAST/STD/SLOW ownership improves risk description but does not create a steadily winning admission process.
5. Entry selection and journey management should remain separate research objects. No Phase-1V output has action or sizing authority.